In [19]:
import numpy as np

import sympy as sp

from sympy import sin
import time
import math

from utils import ( 
        get_fitted_scaler_from_csv, 
        clean_point, 
        symbolic_gradient, 
        mean_absolute_error_numpy, 
        get_distances_and_features_ckdtree,
        calculate_weight_matrix,
        predict_gibbs_energy_arrays,
        COLUMN_TO_VAR
)

# Example usage
file_path = "/home/couchbucks/Documents/saketh/regression/regression/718_all/718_all.csv"
target_col = 'Gibbs_energy'
scaler, scaled_df = get_fitted_scaler_from_csv(file_path, target_col)


# Calculate the number of rows to sample (80% of the DataFrame)
sample_size = int(len(scaled_df) * 0.8)

# Randomly sample 80% of the rows
reference_df = scaled_df.sample(n=sample_size, random_state=42)

# Specify the new order of columns
new_order = [
    'Temperature_[K]',
    'Mass_fraction_Nb',
    'Mass_fraction_Al',
    'Mass_fraction_Ti',
    'Mass_fraction_Mo',
    'Mass_fraction_Fe',
    'Mass_fraction_Cr',
    # 'Mass_fraction_Ni'
]


ref_gibbs = np.array(reference_df['Gibbs_energy'])

reference_df_rename = reference_df[new_order].rename(columns=COLUMN_TO_VAR)

df_reset_ref = reference_df_rename.reset_index(drop=True)

# Convert the DataFrame to a NumPy array
ref_points = np.array([clean_point(point) for point in df_reset_ref.values])


sampled_indices = reference_df.index

# Select the remaining 20% of the rows that were not sampled
remaining_df = scaled_df.drop(sampled_indices)

test_gibbs = np.array(remaining_df['Gibbs_energy'])

# Rename columns and reorder for consistency
remaining_df = remaining_df[new_order].rename(columns=COLUMN_TO_VAR)

df_reset = remaining_df.reset_index(drop=True)

# Convert the DataFrame to a NumPy array
query_points = np.array([clean_point(point) for point in df_reset.values])

# Function to compute the symbolic gradient of an expression with respect to each variable

x0, x1, x2, x3, x4, x5, x6 = sp.symbols('x0 x1 x2 x3 x4 x5 x6')
filter_variables = [x0, x1, x2, x3, x4, x5, x6]


relevant_variables = [x0, x1, x2, x3, x4, x5, x6]

# 0.00041702 ± 0.00228150
SIMPLIFIED_EQS = {
    'nb': 1.0327**(x0 - 0.1473) * (-x0 + 0.0214 * x1**2 - 0.0892 * x1 + 0.0117),
    'al': 0.1710 * 0.4880**x2 - 0.951 * x0 - 0.0346 * x0**2 + 0.0132 * x0 * x2 - 0.0012 * x2**2 - 0.414,
    'ti': -x0 + 0.244 * (0.5498**x3 - 1.0615),
    'mo': -x0 - 0.0321 * x0**2 + 0.0321 * sin(x4) + 0.0487,
    'fe': 1.0334572**x0 * (-x0),  # Simplified with assumptions
    # 'fe': -x0 + 0.0323 * x0**2 + 0.00302 * x0 * x5 + 0.00086 * x5**2 - 0.0323 * exp(-0.1797 * x5),
    # 'cr': -x0 - x0**2 * 0.0323 + x6 * sin(sin(0.8701099**(x0 - x6))) * 0.0323 + 0.0305
    'cr': -x0 - 0.03286*x0**2 - 0.0241007*(-0.04523*x6**2 - x6) + 0.032916
}

EVEN_MORE_SIMPLIFIED_EQS = {
    'nb': 1.03 * (-x0 + 0.0214 * x1**2 - 0.089 * x1 + 0.0117),
    'al': 0.171 * 0.5**x2 - x0 - 0.035 * x0**2 + 0.013 * x0 * x2 - 0.0012 * x2**2 - 0.414,
    'ti': -x0 + 0.244 * (0.55**x3 - 1.06),
    'mo': -x0 - 0.032 * x0**2 + 0.032 * sin(x4) + 0.05,
    'fe': 1.03* (-x0),  # Simplified with assumptions
    # 'fe': -x0 + 0.0323 * x0**2 + 0.00302 * x0 * x5 + 0.00086 * x5**2 - 0.0323 * exp(-0.1797 * x5),
    # 'cr': -x0 - x0**2 * 0.0323 + x6 * sin(sin(0.8701099**(x0 - x6))) * 0.0323 + 0.0305
    'cr': -x0 - 0.033*x0**2 - 0.024*(-0.045*x6**2 - x6) + 0.033
}

EVEN_EVEN_MORE_SIMPLIFIED_EQS = {
    'nb': -x0 + 0.021 * x1**2 - 0.089 * x1 + 0.012,
    'al': 0.17 * 0.5**x2 - x0 - 0.035 * x0**2 + 0.013 * x0 * x2 - 0.001 * x2**2 - sp.sqrt(2),
    'ti': -x0 + 0.244 * (0.5**x3 - 1),
    'mo': -x0 - 0.032 * x0**2 + 0.032 * sin(x4) + 0.05,
    'fe': -x0,  # Simplified with assumptions
    # 'fe': -x0 + 0.0323 * x0**2 + 0.00302 * x0 * x5 + 0.00086 * x5**2 - 0.0323 * exp(-0.1797 * x5),
    # 'cr': -x0 - x0**2 * 0.0323 + x6 * sin(sin(0.8701099**(x0 - x6))) * 0.0323 + 0.0305
    'cr': -x0 - 0.033*x0**2 - 0.025*(-0.045*x6**2 - x6) + 0.033
}

EVEN_EVEN_EVEN_MORE_SIMPLIFIED_EQS = {
    'nb': -x0 + 0.02 * x1**2 - 0.09 * x1 + 0.01,
    'al': 0.17 * 0.5**x2 - x0 - 0.04 * x0**2 + 0.01 * x0 * x2 - 0.001 * x2**2 - sp.sqrt(2),
    'ti': -x0 + 0.25 * (0.5**x3 - 1),
    'mo': -x0 - 0.03 * x0**2 + 0.03 * sin(x4) + 0.05,
    'fe': -x0,  # Simplified with assumptions
    # 'fe': -x0 + 0.0323 * x0**2 + 0.00302 * x0 * x5 + 0.00086 * x5**2 - 0.0323 * exp(-0.1797 * x5),
    # 'cr': -x0 - x0**2 * 0.0323 + x6 * sin(sin(0.8701099**(x0 - x6))) * 0.0323 + 0.0305
    'cr': -x0 - 0.03*x0**2 - 0.03*(-0.05*x6**2 - x6) + 0.03
}

EVEN_EVEN_EVEN_EVEN_MORE_SIMPLIFIED_EQS = {
    'nb': -x0 + (1/50) * x1**2 - (1/10) * x1 + (1/100),
    'al':  (1/6) * 0.5**x2 - x0 - (1/25) * x0**2 + (1/100) * x0 * x2 - (1/1000) * x2**2 - sp.sqrt(2),
    'ti': -x0 + (1/4) * (0.5**x3 - 1),
    'mo': -x0 - (1/3) * x0**2 + (1/3) * sin(x4) + (1/20),
    'fe': -x0,  # Simplified with assumptions
    # 'fe': -x0 + 0.0323 * x0**2 + 0.00302 * x0 * x5 + 0.00086 * x5**2 - 0.0323 * exp(-0.1797 * x5),
    # 'cr': -x0 - x0**2 * 0.0323 + x6 * sin(sin(0.8701099**(x0 - x6))) * 0.0323 + 0.0305
    'cr': -x0 - (1/3)*x0**2 - (1/3)*(-(1/20)*x6**2 - x6) + (1/3)
}

medium_equations = {
    'nb': 1.03271**(x0 - 0.14734)*(-x0 + 0.02140*x1**2 - 0.08920*x1 + 0.01167),
    'al': 0.48800**x2*0.37150 - x0 - 0.03637*(x0 - 0.190697 * x2)**2 - 0.43599,
    'ti': (-x0 + 0.24438*(0.54978**x3 - 1.06153)),
    'mo': -x0 + (-x0**2 + sin(-1.18954**(x0 + 0.99362) + x4) + 1.51626)*0.03205,
    # 'fe': -x0 + 0.23703/(x5 + exp(2)),
    'fe': -x0 + (-(-x0 - 0.17965141*x5*0.25846276)**2 + sp.log(-0.17965141*x5))*0.0322746291149881, # sp.log = exp = e^(x)
    'cr': -x0 - 0.03286*x0**2 - 0.0241007*(-0.04523*x6**2 - x6) + 0.032916
    # Add other elements...
}


full_equations = {
    # 'nb': (-x0 - 0.079403974*x1*(0.73890084**x1 + 0.04306198) + 0.0091454005)/math.sqrt(0.9366797**(x0 - 0.10534109)),
    'nb': (-x0 - 0.079403974*x1*(0.73890084**x1 + 0.04306198) + 0.0091454005)/((0.9366797**(x0 - 0.10534109))**0.5),
    'al': 0.352343650290714*0.4862979**x2 - 0.352343650290714*1.2002983**(sp.log(x0**2)**(1/4)) - 0.950689196092642*x0, # sp.log = exp = e^(x)
    'ti': (0.22641088*0.5306325**x3 - x0 - 0.242776)*((1.0679464**(x0 - 0.3363843*x3 - 0.45122325))**0.5),
    # 'ti': 'ti': (0.22641088*0.5306325**x3 - x0 - 0.242776)*math.sqrt(1.0679464**(x0 - 0.3363843*x3 - 0.45122325)),
    'mo': 1.0331628**x0*(-x0 - 1*(-0.02004116) + 0.0105717776067225*(-x4 - 0.3921473)**2/1.0812141**(2*x0*x4)),
    'fe': 1.0334572**x0*(-x0 + 0.23702882/(x5 + sp.log(2*(1.0334572**x5)**(x0 - 1*(-0.49997815))))),
    'cr': -x0 + (-0.7220202*1.086616**(-x0 + x6)*x6 + x0**2)*(-0.033271) - 1*(-0.032068063)
}

gradients={}


use_equations = EVEN_EVEN_EVEN_EVEN_MORE_SIMPLIFIED_EQS

for key in use_equations:
    gradients[key] = symbolic_gradient(use_equations[key], relevant_variables)
    

print(gradients)

{'nb': [-1, 0.04*x1 - 0.1, 0, 0, 0, 0, 0], 'al': [-0.08*x0 + 0.01*x2 - 1, 0, -0.115524530093324*0.5**x2 + 0.01*x0 - 0.002*x2, 0, 0, 0, 0], 'ti': [-1, 0, 0, -0.173286795139986*0.5**x3, 0, 0, 0], 'mo': [-0.666666666666667*x0 - 1, 0, 0, 0, 0.333333333333333*cos(x4), 0, 0], 'fe': [-1, 0, 0, 0, 0, 0, 0], 'cr': [-0.666666666666667*x0 - 1, 0, 0, 0, 0, 0, 0.0333333333333333*x6 + 0.333333333333333]}


/home/couchbucks/Documents/saketh/regression/regression/utils.py:157: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True)


In [20]:
# Assuming `scaled_ref_data_points` is already defined
n_ref_points = len(ref_points)
n_features = len(relevant_variables)
n_gradients = len(relevant_variables)  # assuming all have the same gradient dimension

# Initialize arrays to store all gradients and gibbs energies
all_gradients = []
ref_gibbs = np.array(reference_df['Gibbs_energy'])


# Populate the arrays
for i, point in enumerate(ref_points):
    # Original point data
    original_ref_str = reference_df.iloc[i]
    
    closest_point = point
    point_dict = {relevant_variables[j]: closest_point[j] for j in range(n_features)}
    
    # Gradient expressions evaluated at the closest point
    gradient_exprs = gradients[original_ref_str.temp_class]
    # print(gradient_exprs)
    # print(point_dict)
    
    gradient_at_closest = np.array([grad.evalf(subs=point_dict) for grad in gradient_exprs], dtype=np.float64)
    
    # Store in the large arrays
    all_gradients.append(gradient_at_closest)
    # all_gibbs_energies[i] = x.gibbs_energy

all_gradients = np.array(all_gradients)

# Assuming 'temp_class' is the column in reference_df indicating temperature class
ref_points_by_class = {}
gibbs_by_class = {}
gradients_by_class = {}

# Loop through each unique temperature class and group the corresponding points
for temp_class in reference_df['temp_class'].unique():
    mask = reference_df['temp_class'] == temp_class
    
    # Filter points that belong to the current temp_class
    temp_class_points = reference_df[mask]
    
    gibbs_by_class[temp_class] = ref_gibbs[mask]
    # print(mask)
    gradients_by_class[temp_class] = all_gradients[mask]
    
    # Extract the points (excluding the target variable column 'Gibbs_energy')
    temp_class_points_values = temp_class_points.drop(columns=['Gibbs_energy', 'temp_class', 'Mass_fraction_Ni']).values # 'Mass_fraction_Ni' becomes NaN after standardization
    
    # Store the points grouped by class
    ref_points_by_class[temp_class] = temp_class_points_values


In [50]:



from scipy.spatial import cKDTree

def get_neighbors(query_points, ref_points_by_class, ref_points, all_gradients, all_gibbs, n, relative=True):
    """
    Get neighbors either per temperature class or globally.
    
    Args:
    - query_points (np.array): Points to query.
    - ref_points_by_class (dict): Reference points grouped by temp_class.
    - ref_points (np.array): All reference points (for absolute neighbors).
    - all_gradients (np.array): Gradients for all reference points.
    - all_gibbs (np.array): Gibbs energies for all reference points.
    - n (int): Number of neighbors to select.
    - relative (bool): True for n neighbors per temp_class, False for n closest overall.
    
    Returns:
    - distances (np.array): Distances to neighbors.
    - closest_points (np.array): Closest points for each query point.
    - gradient_array (np.array): Gradient arrays for the neighbors.
    - gibbs_array (np.array): Gibbs energy arrays for the neighbors.
    """
    if relative:
        # Relative neighbors: n neighbors per temp_class
        all_distances = []
        all_closest_points = []
        all_gradients = []
        all_gibbs = []

        # print(ref_points_by_class)
        
        KDTrees = {}
        
        for temp_class, points in ref_points_by_class.items():
            
            # print(points.shape)
            
            tree = cKDTree(points)
            KDTrees[temp_class] = tree
        
        print(len(query_points))

        for q_point in query_points:
            
            distances_append = []
            
            closest_points_append = []
            gradients_append = []
            gibbs_append = []
            
            for temp_class, ref_points in ref_points_by_class.items():
                # Build KDTree for each temp_class
                # tree = cKDTree(points)
                
                tree = KDTrees[temp_class]
                
                # Query KDTree for the closest n neighbors
                distances, indices = tree.query(q_point, k=n)
                
                # print(indices)
                
                # break
                
                # Collect neighbors, gradients, and Gibbs energies
                distances_append.extend(distances)
                closest_points_append.extend(ref_points[indices])
                gradients_append.extend(gradients_by_class[temp_class][indices])
                gibbs_append.extend(gibbs_by_class[temp_class][indices])
            
            all_distances.append(distances_append)
            all_closest_points.append(closest_points_append)
            all_gradients.append(gradients_append)
            all_gibbs.append(gibbs_append)
        
        # Combine results across classes
        distances = np.array(all_distances)  # Combine distances
        closest_points = np.array(all_closest_points)  # Combine closest points
        gradient_array = np.array(all_gradients)  # Combine gradients
        gibbs_array = np.array(all_gibbs)  # Combine Gibbs energies

    else:
        # Absolute neighbors: n closest overall
        tree = cKDTree(ref_points)
        distances, indices = tree.query(query_points, k=n)
        
        closest_points = ref_points[indices]
        gradient_array = all_gradients[indices]
        gibbs_array = all_gibbs[indices]

    return distances, closest_points, gradient_array, gibbs_array




NUMBER_OF_NEIGHBORS = 3

start = time.perf_counter()

relative_neighbors=True

# Use `ref_points_by_class` for relative neighbors
distances, closest_points, gradient_array, gibbs_array = get_neighbors(
    query_points=query_points,
    ref_points_by_class=ref_points_by_class,
    ref_points=ref_points,  # If using absolute neighbors, pass the full set of points
    all_gradients=all_gradients,
    all_gibbs=ref_gibbs,
    n=NUMBER_OF_NEIGHBORS,
    relative=True
)


# distances, closest_points, gradient_array, gibbs_array = get_distances_and_features_ckdtree(query_points, 
#                                                                                             ref_points, 
#                                                                                             n=NUMBER_OF_NEIGHBORS,
#                                                                                             all_gradients=all_gradients,
#                                                                                             all_gibbs_energies=ref_gibbs
#                                                                                             )



1637


(708, 7)


In [52]:
distances.shape

(1637, 18)

In [53]:
weights = calculate_weight_matrix(distances)

In [54]:
print(query_points.shape)
print(closest_points.shape)

print(gradient_array.shape)
print(gibbs_array.shape)



(1637, 7)
(1637, 18, 7)
(1637, 18, 7)
(1637, 18)


In [55]:
print(weights.shape)

(1637, 18)


In [56]:
predictions = predict_gibbs_energy_arrays(
    query_points, closest_points, gradient_array, gibbs_array, weights, weighted=True
)


In [17]:
import pandas as pd

def find_best_models(file_path):
    """
    Finds the models with the lowest MSE and MAE from a given CSV file.
    
    Args:
        file_path (str): Path to the CSV file containing model data.
    
    Returns:
        tuple: Two dictionaries, one for the lowest MSE model and one for the lowest MAE model.
    """
    # Read the CSV file
    data = pd.read_csv(file_path)
    
    # Find the row with the lowest MSE
    best_mse_row = data.loc[data['MSE'].idxmin()]
    
    # Find the row with the lowest MAE
    best_mae_row = data.loc[data['MAE'].idxmin()]
    
    best_mape_row = data.loc[data['MAPE'].idxmin()]
    
    # Prepare the output as dictionaries
    best_mse_model = {
        'Sample': best_mse_row['Sample'],
        'MSE': best_mse_row['MSE'],
        'MAE': best_mse_row['MAE'],
        'MAPE': best_mse_row['MAPE'],
        'Equation': best_mse_row['Equation']
    }
    best_mae_model = {
        'Sample': best_mae_row['Sample'],
        'MSE': best_mae_row['MSE'],
        'MAE': best_mae_row['MAE'],
        'MAPE': best_mae_row['MAPE'],
        'Equation': best_mae_row['Equation']
    }
    
    best_mape_model = {
        'Sample': best_mape_row['Sample'],
        'MSE': best_mape_row['MSE'],
        'MAE': best_mape_row['MAE'],
        'MAPE': best_mape_row['MAPE'],
        'Equation': best_mape_row['Equation']
    }
    
    return best_mse_model, best_mae_model, best_mape_model

# Example usage
file_path = '/home/couchbucks/Documents/saketh/regression/regression/718_all_cr/linear_regression/linear_regression_metrics_1.csv'
best_mse, best_mae, best_mape_model = find_best_models(file_path)
print("Best MSE Model:", best_mse)
print("Best MAE Model:", best_mae)
print(f"Best MAPE Model: {best_mape_model}")



Best MSE Model: {'Sample': 2, 'MSE': 368909.4042194189, 'MAE': 516.5547069927487, 'MAPE': 0.0064084199043126, 'Equation': '41465.7903991173 – 94.5441072023x1 + 16736.057976256x2'}
Best MAE Model: {'Sample': 2, 'MSE': 368909.4042194189, 'MAE': 516.5547069927487, 'MAPE': 0.0064084199043126, 'Equation': '41465.7903991173 – 94.5441072023x1 + 16736.057976256x2'}
Best MAPE Model: {'Sample': 2, 'MSE': 368909.4042194189, 'MAE': 516.5547069927487, 'MAPE': 0.0064084199043126, 'Equation': '41465.7903991173 – 94.5441072023x1 + 16736.057976256x2'}


### Timing for training PySR

In [ ]:
# took 3926 sec for 15 equations - 225, 80

In [ ]:
# Time for 5: 2625.240789683012 - 300, 80

### Timing for Linear Regression

In [ ]:
# Total Time: 0.028301957005169243s for 15 samples